In [ ]:
try:
    import pyspark.sql.functions as F
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

SCHEMA_ORIGEM = "silver"
SCHEMA_DESTINO = "gold"
TABELA_FILMES_ORIGEM = "tb_info_filmes"
TABELA_GENEROS_ORIGEM = "tb_generos"
TABELA_PESSOAS_ORIGEM = "tb_pessoas_empresas"
TABELA_AVALIACOES_ORIGEM = "tb_avaliacoes_usuarios"

TABELA_DIM_FILMES = "dim_movies"
TABELA_DIM_GENEROS = "dim_genres"
TABELA_DIM_PESSOAS = "dim_people"
TABELA_DIM_PRODUTORAS = "dim_companies"
TABELA_DIM_AVALIACOES = "dim_reviews"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

In [ ]:
# carrega as tabelas Silver e confirma as colunas necessárias antes das dimensões
df_filmes_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_FILMES_ORIGEM}")
df_generos_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_GENEROS_ORIGEM}")
df_pessoas_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_PESSOAS_ORIGEM}")
df_avaliacoes_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_AVALIACOES_ORIGEM}")

colunas_filmes_esperadas = {
    "id_filme", "id_imdb", "titulo", "titulo_original",
    "idioma_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "status", "sinopse", "tagline"
}
colunas_filmes_ausentes = colunas_filmes_esperadas.difference(df_filmes_silver.columns)
if colunas_filmes_ausentes:
    raise ValueError(f"Colunas ausentes na Silver de filmes: {sorted(colunas_filmes_ausentes)}")

# a chave do filme usa o id natural, mantendo o mesmo valor em cada reprocessamento
df_dim_movies = (
    df_filmes_silver
    .where(F.col("id_filme").isNotNull())
    .withColumn("sk_movie_id", F.sha2(F.concat_ws("|", F.lit("movie"), F.col("id_filme").cast("string")), 256))
    .select(
        "sk_movie_id", "id_filme", "id_imdb", "titulo",
        "titulo_original", "idioma_original", "data_lancamento",
        "ano_lancamento", "duracao_minutos", "status",
        "sinopse", "tagline"
    )
)

# grava em overwrite para permitir reprocessamento idempotente da dimensão
(df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_FILMES}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_FILMES}")
display(df_dim_movies.limit(10))

In [ ]:
# deduplica os gêneros sem diferenciar maiúsculas e minúsculas
df_dim_genres = (
    df_generos_silver
    .select(F.trim(F.col("nome_genero")).alias("nome_genero"))
    .where(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .withColumn("nome_genero_normalizado", F.lower(F.col("nome_genero")))
    .groupBy("nome_genero_normalizado")
    .agg(F.first("nome_genero", ignorenulls=True).alias("nome_genero"))
    .withColumn("sk_genre_id", F.sha2(F.concat_ws("|", F.lit("genre"), F.col("nome_genero_normalizado")), 256))
    .select("sk_genre_id", "nome_genero")
)

(df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}")
display(df_dim_genres.orderBy("nome_genero").limit(10))

In [ ]:
# separa pessoas e produtoras para que cada dimensão tenha um único tipo de entidade
df_entidades = (
    df_pessoas_silver
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_entidade"),
        F.trim(F.col("tipo_entidade")).alias("tipo_entidade")
    )
    .where(F.col("nome_entidade").isNotNull() & (F.col("nome_entidade") != ""))
)

df_dim_people = (
    df_entidades
    .where(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .dropDuplicates(["nome_entidade", "tipo_entidade"])
    .withColumn("sk_person_id", F.sha2(F.concat_ws("|", F.lit("person"), F.col("nome_entidade"), F.col("tipo_entidade")), 256))
    .select("sk_person_id", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
)

df_dim_companies = (
    df_entidades
    .where(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .dropDuplicates(["nome_produtora"])
    .withColumn("sk_company_id", F.sha2(F.concat_ws("|", F.lit("company"), F.col("nome_produtora")), 256))
    .select("sk_company_id", "nome_produtora")
)

for df_dimensao, tabela in [
    (df_dim_people, TABELA_DIM_PESSOAS),
    (df_dim_companies, TABELA_DIM_PRODUTORAS),
]:
    (df_dimensao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{SCHEMA_DESTINO}.{tabela}"
    ))
    print(f"Tabela gravada: {SCHEMA_DESTINO}.{tabela}")

display(df_dim_people.limit(10))
display(df_dim_companies.limit(10))

In [ ]:
# agrega avaliações no grão de um registro por filme
df_reviews_agregadas = (
    df_avaliacoes_silver
    .where(F.col("id_filme").isNotNull())
    .groupBy("id_filme")
    .agg(
        F.count(F.lit(1)).cast("BIGINT").alias("quantidade_avaliacoes"),
        F.round(F.avg("nota"), 2).cast("DECIMAL(10,2)").alias("nota_media")
    )
)

# relaciona a avaliação agregada com a chave substituta da dimensão de filmes
df_dim_reviews = (
    df_reviews_agregadas
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .withColumn("sk_review_id", F.sha2(F.concat_ws("|", F.lit("review"), F.col("id_filme").cast("string")), 256))
    .select("sk_review_id", "sk_movie_id", "quantidade_avaliacoes", "nota_media")
)

(df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_AVALIACOES}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_AVALIACOES}")
display(df_dim_reviews.limit(10))

In [ ]:
# valida chaves primárias, deduplicação e relacionamento da dimensão de avaliações
df_validacao_dim_movies = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_FILMES}")
df_validacao_dim_genres = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}")
df_validacao_dim_people = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_PESSOAS}")
df_validacao_dim_companies = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_PRODUTORAS}")
df_validacao_dim_reviews = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_AVALIACOES}")

duplicados_filmes = df_validacao_dim_movies.groupBy("sk_movie_id").count().where(F.col("count") > 1).count()
duplicados_generos = df_validacao_dim_genres.groupBy("sk_genre_id").count().where(F.col("count") > 1).count()
duplicados_pessoas = df_validacao_dim_people.groupBy("sk_person_id").count().where(F.col("count") > 1).count()
duplicados_produtoras = df_validacao_dim_companies.groupBy("sk_company_id").count().where(F.col("count") > 1).count()
duplicados_avaliacoes = df_validacao_dim_reviews.groupBy("sk_review_id").count().where(F.col("count") > 1).count()
avaliacoes_sem_filme = (
    df_validacao_dim_reviews.join(df_validacao_dim_movies.select("sk_movie_id"), on="sk_movie_id", how="left_anti").count()
)

if any(valor != 0 for valor in [duplicados_filmes, duplicados_generos, duplicados_pessoas, duplicados_produtoras, duplicados_avaliacoes, avaliacoes_sem_filme]):
    raise AssertionError(
        "As dimensões possuem chaves duplicadas ou avaliações sem filme correspondente."
    )

print(f"Filmes: {df_validacao_dim_movies.count()} | Gêneros: {df_validacao_dim_genres.count()}")
print(f"Pessoas: {df_validacao_dim_people.count()} | Produtoras: {df_validacao_dim_companies.count()}")
print(f"Filmes com avaliações agregadas: {df_validacao_dim_reviews.count()}")
print("Validação de chaves e relacionamentos concluída sem inconsistências.")